In [ ]:
!pip install ctgan

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.7/66.7 kB 6.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 60.6 MB/s eta 0:00:00


In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
from sklearn.model_selection import train_test_split
import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler
from ctgan import CTGAN
from sklearn.metrics import mean_squared_error

In [ ]:
data = pd.read_csv('train_feature_v5.csv')

In [ ]:
# Prepare dataset for VAE
X = data.drop(columns=["price"]).values  # Exclude target variable for input
y = data["price"].values  # Target variable

In [ ]:
# Normalize features and target variable
scaler_X = StandardScaler()
X_normalized = scaler_X.fit_transform(X)
scaler_y = StandardScaler()
y_normalized = scaler_y.fit_transform(y.reshape(-1, 1)).flatten()

In [ ]:
# Convert to tensors
X_tensor = torch.tensor(X_normalized, dtype=torch.float32)
y_tensor = torch.tensor(y_normalized, dtype=torch.float32).view(-1, 1)

# Combine X and y for the CVAE model
dataset = TensorDataset(X_tensor, y_tensor)
dataloader = DataLoader(dataset, batch_size=32, shuffle=True)

# VAE for Synthetic Data Generation

In [ ]:
# Define Conditional VAE model
class CVAE(nn.Module):
    def __init__(self, input_dim, latent_dim=32, hidden_dim=64):
        super(CVAE, self).__init__()
        # Encoder
        self.fc1 = nn.Linear(input_dim + 1, hidden_dim)  # +1 to include price as a condition
        self.fc2_mu = nn.Linear(hidden_dim, latent_dim)
        self.fc2_logvar = nn.Linear(hidden_dim, latent_dim)
        # Decoder
        self.fc3 = nn.Linear(latent_dim + 1, hidden_dim)  # +1 to include price as a condition
        self.fc4 = nn.Linear(hidden_dim, input_dim)

    def encode(self, x, y):
        # Concatenate feature and price as input to the encoder
        x_cond = torch.cat([x, y], dim=1)
        h = torch.relu(self.fc1(x_cond))
        mu = self.fc2_mu(h)
        logvar = self.fc2_logvar(h)
        return mu, logvar

    def reparameterize(self, mu, logvar):
        std = torch.exp(0.5 * logvar)
        eps = torch.randn_like(std)
        return mu + eps * std

    def decode(self, z, y):
        # Concatenate latent vector and price as input to the decoder
        z_cond = torch.cat([z, y], dim=1)
        h = torch.relu(self.fc3(z_cond))
        return self.fc4(h)

    def forward(self, x, y):
        mu, logvar = self.encode(x, y)
        z = self.reparameterize(mu, logvar)
        return self.decode(z, y), mu, logvar

# Define the conditional VAE loss function
def cvae_loss(recon_x, x, mu, logvar):
    recon_loss = nn.functional.mse_loss(recon_x, x, reduction="mean")
    kld_loss = -0.5 * torch.mean(1 + logvar - mu.pow(2) - logvar.exp())
    return recon_loss + kld_loss

# Initialize CVAE, optimizer, and loss function
input_dim = X_tensor.shape[1]
latent_dim = 32
hidden_dim = 64
cvae = CVAE(input_dim, latent_dim, hidden_dim)
optimizer = optim.Adam(cvae.parameters(), lr=0.001)

# Train the CVAE model
epochs = 100
cvae.train()
for epoch in range(epochs):
    total_loss = 0
    for x_batch, y_batch in dataloader:
        optimizer.zero_grad()
        recon_x, mu, logvar = cvae(x_batch, y_batch)
        loss = cvae_loss(recon_x, x_batch, mu, logvar)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    print(f"Epoch {epoch+1}/{epochs}, Loss: {total_loss / len(dataloader.dataset):.4f}")

Epoch 1/100, Loss: 0.0265
Epoch 2/100, Loss: 0.0227
Epoch 3/100, Loss: 0.0207
Epoch 4/100, Loss: 0.0194
Epoch 5/100, Loss: 0.0186
Epoch 6/100, Loss: 0.0178
Epoch 7/100, Loss: 0.0172
Epoch 8/100, Loss: 0.0168
Epoch 9/100, Loss: 0.0165
Epoch 10/100, Loss: 0.0162
Epoch 11/100, Loss: 0.0159
Epoch 12/100, Loss: 0.0157
Epoch 13/100, Loss: 0.0156
Epoch 14/100, Loss: 0.0154
Epoch 15/100, Loss: 0.0152
Epoch 16/100, Loss: 0.0151
Epoch 17/100, Loss: 0.0150
Epoch 18/100, Loss: 0.0149
Epoch 19/100, Loss: 0.0148
Epoch 20/100, Loss: 0.0147
Epoch 21/100, Loss: 0.0146
Epoch 22/100, Loss: 0.0144
Epoch 23/100, Loss: 0.0144
Epoch 24/100, Loss: 0.0143
Epoch 25/100, Loss: 0.0142
Epoch 26/100, Loss: 0.0141
Epoch 27/100, Loss: 0.0140
Epoch 28/100, Loss: 0.0140
Epoch 29/100, Loss: 0.0139
Epoch 30/100, Loss: 0.0138
Epoch 31/100, Loss: 0.0138
Epoch 32/100, Loss: 0.0137
Epoch 33/100, Loss: 0.0137
Epoch 34/100, Loss: 0.0136
Epoch 35/100, Loss: 0.0135
Epoch 36/100, Loss: 0.0134
Epoch 37/100, Loss: 0.0135
Epoch 38/1

In [ ]:
# Generate synthetic data
cvae.eval()
num_synthetic_samples = 25000
with torch.no_grad():
    # Sample random latent vectors and generate corresponding synthetic features and prices
    z = torch.randn(num_synthetic_samples, latent_dim)
    y_synthetic_normalized = torch.tensor(np.random.choice(y_normalized, num_synthetic_samples), dtype=torch.float32).view(-1, 1)
    synthetic_data = cvae.decode(z, y_synthetic_normalized).numpy()

# Reverse normalization for synthetic data
synthetic_data = scaler_X.inverse_transform(synthetic_data)
synthetic_prices = scaler_y.inverse_transform(y_synthetic_normalized.numpy())

# Combine into a DataFrame
import pandas as pd
synthetic_data_df = pd.DataFrame(synthetic_data, columns=data.columns.drop("price"))
synthetic_data_df["price"] = synthetic_prices.flatten()

# Combine with original data
augmented_data = pd.concat([data, synthetic_data_df], ignore_index=True).sample(frac=1).reset_index(drop=True)

In [ ]:
augmented_data.to_csv('augmented_data_vae.csv', index=False)

# Using CTGAN for Synthetic Data Generation

In [ ]:
categorical_columns = ['feature_manufactured_vehicle_age',
                        'feature_is_electric',
                        'feature_make_rolls-royce',
                        'feature_make_lotus',
                        'feature_make_ferrari',
                        'feature_make_bentley',
                        'feature_make_porsche',
                        'feature_make_mclaren',
                        'feature_make_bmw',
                        'feature_make_mercedes-benz',
                        'feature_make_lamborghini',
                        'feature_make_nissan',
                        'feature_make_honda',
                        'feature_make_jaguar',
                        'feature_make_toyota',
                        'feature_make_mitsubishi',
                        'feature_make_target_encoded',
                        'feature_low_mileage',
                        'feature_fuel_label',
                        'feature_fuel_type_diesel',
                        'feature_fuel_type_diesel-electric',
                        'feature_fuel_type_electric',
                        'feature_fuel_type_petrol',
                        'feature_fuel_type_petrol-electric',
                        'feature_no_of_owners',
                        'feature_type',
                        'feature_ownership',
                        'feature_condition',
                        'feature_special_features',
                        'feature_sales',
                        'feature_type_of_vehicle',
                        'feature_price_category'
                        ]

In [ ]:
model = CTGAN()
model.fit(data, discrete_columns=categorical_columns)

/usr/local/lib/python3.10/dist-packages/torch/autograd/graph.py:825: UserWarning: Attempting to run cuBLAS, but there was no current CUDA context! Attempting to set the primary context... (Triggered internally at ../aten/src/ATen/cuda/CublasHandlePool.cpp:135.)
  return Variable._execution_engine.run_backward(  # Calls into the C++ engine to run the backward pass


In [ ]:
synthetic_data = model.sample(25000)

In [ ]:
augmented_data = pd.concat([data, synthetic_data], ignore_index=True).sample(frac=1).reset_index(drop=True)

In [ ]:
augmented_data.to_csv('augmented_data_ctgan.csv', index=False)

# Model Used to test the Synthetic Data Generated

In [25]:
# Gated Linear Units
class PricePredictionNN(nn.Module):
    def __init__(self, input_dim):
        super(PricePredictionNN, self).__init__()
        self.fc1 = nn.Linear(input_dim, 128)
        self.glu1 = nn.GLU()
        self.fc2 = nn.Linear(64, 64)  # Output of GLU is half the previous layer
        self.glu2 = nn.GLU()
        self.output = nn.Linear(32, 1)  # Single output for regression

    def forward(self, x):
        x = self.glu1(self.fc1(x))
        x = self.glu2(self.fc2(x))
        return self.output(x)

In [26]:
# Training function with early stopping
def train_model(model, train_loader, val_loader, criterion, optimizer, epochs=20):
    global best_val_loss, epochs_no_improve
    for epoch in range(epochs):
        model.train()
        train_loss = 0.0
        for inputs, targets in train_loader:
            optimizer.zero_grad()
            outputs = model(inputs)
            loss = criterion(outputs, targets)
            loss.backward()
            optimizer.step()
            train_loss += loss.item() * inputs.size(0)

        # Validation phase
        model.eval()
        val_loss = 0.0
        with torch.no_grad():
            for inputs, targets in val_loader:
                outputs = model(inputs)
                loss = criterion(outputs, targets)
                val_loss += loss.item() * inputs.size(0)

        # Calculate average losses
        train_loss /= len(train_loader.dataset)
        val_loss /= len(val_loader.dataset)

        print(f"Epoch {epoch+1}/{epochs}, Train Loss: {train_loss:.4f}, Val Loss: {val_loss:.4f}")

        # Early stopping check
        if val_loss < best_val_loss:
            best_val_loss = val_loss
            epochs_no_improve = 0
            best_model = model.state_dict()  # Save the best model's parameters
        else:
            epochs_no_improve += 1
            if epochs_no_improve >= patience:
                print(f"Early stopping triggered after {epoch+1} epochs.")
                model.load_state_dict(best_model)  # Load the best model parameters
                break

# Testing Data generated using VAE using NN model (GLU)

In [13]:
augmented_data = pd.read_csv('augmented_data_vae.csv')

In [14]:
# Separate features and target
X = augmented_data.drop(columns=["price"]).values
y = augmented_data["price"].values

In [15]:
# Normalize features and target for stability
scaler_X = StandardScaler()
X_normalized = scaler_X.fit_transform(X)
scaler_y = StandardScaler()
y_normalized = scaler_y.fit_transform(y.reshape(-1, 1)).flatten()

In [16]:
# Split into training and validation sets
X_train, X_val, y_train, y_val = train_test_split(X_normalized, y_normalized, test_size=0.2, random_state=42)

# Convert to PyTorch tensors
X_train_tensor = torch.tensor(X_train, dtype=torch.float32)
y_train_tensor = torch.tensor(y_train, dtype=torch.float32).view(-1, 1)
X_val_tensor = torch.tensor(X_val, dtype=torch.float32)
y_val_tensor = torch.tensor(y_val, dtype=torch.float32).view(-1, 1)

# Create DataLoaders for training and validation
train_dataset = TensorDataset(X_train_tensor, y_train_tensor)
val_dataset = TensorDataset(X_val_tensor, y_val_tensor)
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False)

In [27]:
# Early stopping configuration
# Initialize the model, loss function, and optimizer
input_dim = X_train.shape[1]
model = PricePredictionNN(input_dim)
criterion = nn.MSELoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

patience = 10  # Number of epochs to wait for improvement
best_val_loss = float('inf')
epochs_no_improve = 0

In [18]:
# Train the model with early stopping
train_model(model, train_loader, val_loader, criterion, optimizer, epochs=100)

In [35]:
import numpy as np
from sklearn.metrics import mean_squared_error

# Ensure model is in evaluation mode
model.eval()

# Make predictions on the validation set
with torch.no_grad():
    val_predictions = model(X_val_tensor).numpy().flatten()
    val_targets = y_val_tensor.numpy().flatten()

# Calculate RMSE on the validation set (converting back from normalized to original scale)
val_predictions_original_scale = scaler_y.inverse_transform(val_predictions.reshape(-1, 1)).flatten()
val_targets_original_scale = scaler_y.inverse_transform(val_targets.reshape(-1, 1)).flatten()

# Calculate RMSE
rmse = np.sqrt(mean_squared_error(val_targets_original_scale, val_predictions_original_scale))
print(f"Validation RMSE: {rmse:.4f}")

Validation RMSE: 23529.0470


In [29]:
test_df = pd.read_csv('sample_data/test_feature_v5.csv')
X_test_normalized = scaler_X.fit_transform(test_df.values)
X_test_tensor = torch.tensor(X_test_normalized, dtype=torch.float32)

test_dataset = TensorDataset(X_test_tensor)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)

In [30]:
predictions = []
model.eval()
with torch.no_grad():
    for inputs in test_loader:
        outputs = model(inputs[0])
        predictions.extend(outputs.numpy().flatten())

In [32]:
predictions = scaler_y.inverse_transform(np.array(predictions).reshape(-1, 1)).flatten()

In [33]:
test_results = pd.DataFrame({'Id': range(len(predictions)), 'Predicted': predictions})

In [34]:
test_results.to_csv('submission_vae.csv', index=False)

# Testing Data generated using CTGAN using NN model (GLU)

In [ ]:
augmented_data = pd.read_csv('augmented_data_ctgan.csv')

In [ ]:
# Separate features and target
X = augmented_data.drop(columns=["price"]).values
y = augmented_data["price"].values

In [ ]:
# Normalize features and target for stability
scaler_X = StandardScaler()
X_normalized = scaler_X.fit_transform(X)
scaler_y = StandardScaler()
y_normalized = scaler_y.fit_transform(y.reshape(-1, 1)).flatten()

In [ ]:
# Split into training and validation sets
X_train, X_val, y_train, y_val = train_test_split(X_normalized, y_normalized, test_size=0.2, random_state=42)

# Convert to PyTorch tensors
X_train_tensor = torch.tensor(X_train, dtype=torch.float32)
y_train_tensor = torch.tensor(y_train, dtype=torch.float32).view(-1, 1)
X_val_tensor = torch.tensor(X_val, dtype=torch.float32)
y_val_tensor = torch.tensor(y_val, dtype=torch.float32).view(-1, 1)

# Create DataLoaders for training and validation
train_dataset = TensorDataset(X_train_tensor, y_train_tensor)
val_dataset = TensorDataset(X_val_tensor, y_val_tensor)
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False)

In [ ]:
# Early stopping configuration
# Initialize the model, loss function, and optimizer
input_dim = X_train.shape[1]
model = PricePredictionNN(input_dim)
criterion = nn.MSELoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

patience = 10  # Number of epochs to wait for improvement
best_val_loss = float('inf')
epochs_no_improve = 0

In [ ]:
# Train the model with early stopping
train_model(model, train_loader, val_loader, criterion, optimizer, epochs=100)

Epoch 1/100, Train Loss: 0.2206, Val Loss: 0.1820
Epoch 2/100, Train Loss: 0.1903, Val Loss: 0.1896
Epoch 3/100, Train Loss: 0.1826, Val Loss: 0.1928
Epoch 4/100, Train Loss: 0.1771, Val Loss: 0.1845
Epoch 5/100, Train Loss: 0.1718, Val Loss: 0.1721
Epoch 6/100, Train Loss: 0.1659, Val Loss: 0.1914
Epoch 7/100, Train Loss: 0.1579, Val Loss: 0.1832
Epoch 8/100, Train Loss: 0.1531, Val Loss: 0.1809
Epoch 9/100, Train Loss: 0.1477, Val Loss: 0.1834
Epoch 10/100, Train Loss: 0.1404, Val Loss: 0.1928
Epoch 11/100, Train Loss: 0.1359, Val Loss: 0.1959
Epoch 12/100, Train Loss: 0.1290, Val Loss: 0.1901
Epoch 13/100, Train Loss: 0.1264, Val Loss: 0.1933
Epoch 14/100, Train Loss: 0.1199, Val Loss: 0.1973
Epoch 15/100, Train Loss: 0.1144, Val Loss: 0.2086
Early stopping triggered after 15 epochs.


In [ ]:
import numpy as np
from sklearn.metrics import mean_squared_error

# Ensure model is in evaluation mode
model.eval()

# Make predictions on the validation set
with torch.no_grad():
    val_predictions = model(X_val_tensor).numpy().flatten()
    val_targets = y_val_tensor.numpy().flatten()

# Calculate RMSE on the validation set (converting back from normalized to original scale)
val_predictions_original_scale = scaler_y.inverse_transform(val_predictions.reshape(-1, 1)).flatten()
val_targets_original_scale = scaler_y.inverse_transform(val_targets.reshape(-1, 1)).flatten()

# Calculate RMSE
rmse = np.sqrt(mean_squared_error(val_targets_original_scale, val_predictions_original_scale))
print(f"Validation RMSE: {rmse:.4f}")

Validation RMSE: 85655.0469


In [ ]:
test_df = pd.read_csv('test_feature_v5.csv')
X_test_normalized = scaler_X.fit_transform(test_df.values)
X_test_tensor = torch.tensor(X_test_normalized, dtype=torch.float32)

test_dataset = TensorDataset(X_test_tensor)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)

In [ ]:
predictions = []
model.eval()
with torch.no_grad():
    for inputs in test_loader:
        outputs = model(inputs[0])
        predictions.extend(outputs.numpy().flatten())

In [ ]:
predictions = scaler_y.inverse_transform(np.array(predictions).reshape(-1, 1)).flatten()

In [ ]:
test_results = pd.DataFrame({'Id': range(len(predictions)), 'Predicted': predictions})

In [ ]:
test_results.to_csv('submission_ctgan.csv', index=False)